# CryptoHypothesisLab — 1/2: Priprema Development paketa

**Samo priprema, ne istraživanje.** Ovaj notebook koristi ugrađeni Google Drive mount na tvom Gmailu. U ovoj sesiji Drive je dostupan kodu; koristi samo zvanični `CryptoHypothesisLab` kod. **Nikad ne otvaraj analitički notebook u ovoj sesiji.**

- Kod javno na GitHubu. Izvorni C1/C2 i privatni rezultati ostaju izvan GitHuba.
- Frozen metadata + SHA-256 provjera, samo mjesec u **2020–2024**, `READ_STARTED` na zasebnom privatnom `research-events` folderu.
- Rezultat je mali ZIP paket minutnih značajki u lokalnom `/content`. Preuzmi ga na računar i zatim **Runtime → Disconnect and delete runtime**.
- FUSE `READ_STARTED` zapis je vlasnički operativni trag, **nije nezavisna/atomska cloud evidencija** niti zamjena za produkcijski potpisani broker. Ovaj postupak je manuelni pilot, samo uz izričito vlasničko odobrenje.
- Google Colab može imati promjenjive RAM/disk limite. Ako prijenos/otključavanje zataji, ne prikazujemo djelimične rezultate.


In [ ]:
# 1. Instaliraj samo javni testirani kod, prije mounta.
import subprocess, sys
from pathlib import Path
repo=Path('/content/hypolab-code')
if repo.exists():
    raise RuntimeError('Koristi novu cistu Colab sesiju; prethodni checkout postoji.')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
subprocess.run([sys.executable,'-m','pip','-q','install','pyarrow==26.0.0'],check=True)
sys.path.insert(0,str(repo/'src'))
print('Kod spreman, commit:',commit[:12])

In [ ]:
# 2. Ugradjeni Colab Drive mount: NEMA vlastitog OAuth klijenta.
from google.colab import drive
from hypolab.frozen import FrozenIndex
from hypolab.colab_prep import prepare_one_development_month
from hypolab.airgap_bundle import inspect_bundle
import secrets, os

drive.mount('/content/drive')
drive_root=Path('/content/drive/MyDrive')
metadata_parent=drive_root/'CryptoResearchLab'/'BTC-DATA-UNIVERSE-V1-REVIEW-20261108'/'BTC_DATA_UNIVERSE_V1_FROZEN'
source_generations=drive_root/'CryptoResearchLab'/'data'/'canonical'/'generations'
private_events=drive_root/'CryptoHypothesisLab'/'research-events'

# Unesi 64-znakovne SHA-256 identitete iz vlastitog frozen dokaza.
universe_sha=input('Frozen Universe SHA-256 (64 znaka): ').strip()
calendar_sha=input('Frozen calendar SHA-256 (64 znaka): ').strip()
metadata_root=metadata_parent/universe_sha
frozen=FrozenIndex.load(metadata_root,universe_sha,calendar_sha)
print('Frozen metadata PASS. Ukupne C1/C2 particije:',sum(k[0]=='klines' for k in frozen.partitions),sum(k[0]=='aggTrades' for k in frozen.partitions))
print('Nijedan market-data red nije otvoren.')

In [ ]:
# 3. Vlasnicka, jedinstvena dozvola za JEDAN Development mjesec.
# Prije ovog koraka potvrdi da prihvatas manuelni Colab pilot; on ima
# slabiju dokazivu trajnost od zasebnog potpisanog owner-brokera.
month=input('Mjesec za pripremu, npr. 2024-07: ').strip()
assert month in (f'{y}-{m:02d}' for y in range(2020,2025) for m in range(1,13)), 'Validation/OOS DENIED'
confirmation=input(f'Upisi tacno AUTHORIZE_DEVELOPMENT_{month}: ').strip()
attempt_id=secrets.token_hex(8)
private_cache=Path('/content/hypolab-cache');private_cache.mkdir(mode=0o700,exist_ok=False)
packet_folder=Path('/content/hypolab-packets');packet_folder.mkdir(mode=0o700,exist_ok=False)
result=prepare_one_development_month(
    metadata_root=metadata_root, generations_root=source_generations,
    events_root=private_events,private_cache_parent=private_cache,
    month=month,attempt_id=attempt_id,confirmation=confirmation,
    expected_universe_sha256=universe_sha,expected_calendar_sha256=calendar_sha,
    assumed_latency_ms=1500,producer_code_commit=commit)
print('Priprema PASS; broj Development minuta:',result.feature_rows)
print('ZIP SHA-256:',result.sha256)
print('Event SHA-256:',result.start_event_sha256)
# U slucaju prekida, start-event ostaje. Ne pokretati automatski ponovo.


In [ ]:
# 4. Paket preuzmi na racunar. U novoj Colab sesiji koristi notebook 02.
from google.colab import files
files.download(str(result.path))
print('Nakon preuzimanja: Runtime > Disconnect and delete runtime.')
